## Пэт-проект: Бюджетирование семейных финансов.
### Стек технологий:  python, jupyter notebook, pandas, numpy, regular expressions, excel, ввод/вывод файлов excel в python, макросы и выпадающие списки excel.
### Что сделано: Создана методика и набор программ на python и excel для бюджетирования и прогнозирования семейных финансов.
### Основные подходы: Анализ ведется на ежемесячной основе по банковским выпискам и данным по операциям с наличными денежными средствами.
### Учет ведется по денежным потокам. Такой подход позволяет контролировать ликвидность и удобен для прогнозирования семейного бюджета.
### Сумма оборотов за месяц равна денежному потоку CF (cash flow).
### С другой стороны на последнее число месяца определяется сумма чистых активов (сумма остатков по дебетовым картам и наличным денежным средствам за вычетом долгов по кредитным картам, долгов по кредитам и прочим долгам). Прирост чистых активов за месяц равен финансовому результату PNL (profit and loss)
### Такой двойной учет позволяет делать выверку операций и, в частности, определять сумму неучтенных операций с наличными денежными средствами.
### Для удобства и унификации операций все доходные и расходные операции относятся к определенной группе, составляя классификатор операций.
### В свою очередь все группы операций делятся на 4 раздела расходных операций:
- ### ежемесячные условно-постоянные (комм.пл, подписки, Б %, Б усл, кредиты),
- ### ежемесячные условно-переменные (супер.м, ReCa, маркет.п, Од&О, Кр&З, мед&ап, бензин, авто, TAXI, трансп, дет&канц, дом&рем, развл&экс),
- ### крупные разовые - быт.т&под, спорт.т, отдых, а/б ж.д,
- ### персональные членов семьи (хобби, фитнес и спорт, обучение, тренеры и репетиторы),
- ### невыясненные и прочие – имя_1, имя_2, …, имя_N, ZOO, невыясн, проч,
### и раздел Доходов:
- ### аренда, з/п, бонус.

### Методика состоит из следующих этапов:
- ### ежемесячное скачивание банковских выписок и оборотных ведомостей в excel (Альфа банк - alfa_05_26.xlsm, T-Банк и др. ) и pdf (Сбербанк - sber_05_26.pdf и др.) форматах,
- ### автоматическая обработка выписок и приведение их к excel файлам единого образца с разнесением расходов по группам операций (программы на python с учетом специфики банков и формата выписок, использование матрицы регулярных выражений для разнесения операций), 


### Исполним файл exl_alfa для преобразования выписки alfa_05_26.xlsm в excel файл заданного образца alfa.xlsm


In [15]:
directory_path = r""
#directory_out = r""
directory_out = r""
sheet_name = "alfa"
file_exl = sheet_name + '_05_26.xlsx'

#pip install PyPDF2
import PyPDF2
import pandas as pd
import numpy as np
import re # импортируем модуль re для работы с регулярными выражениями, в чистом Python их нет

# Открываем excel-файл 
#file_exl = directory_path + file_exl
file_exl = directory_path + file_exl
with open(file_exl, 'rb') as file:
    df = pd.read_excel(file)   
df = df[['Дата операции', 'Описание операции', 'Категория', 'Сумма', 'Тип']].rename(columns =
                                                                    {'Дата операции'     : 'Дата',
                                                                    'Описание операции'  : 'Назначение платежа',
                                                                     'Сумма'             : 'Сумма платежа'
                                                                     })
#присвоение знака сумме платежа                                                                           
df['Сумма платежа'] = (-1 + 2 * (df['Тип'] == 'Пополнение').astype('int')) * df['Сумма платежа']
#df['Сумма платежа'] = df['Сумма платежа'].apply(lambda x: str(x).replace('.',','))

#создание словаря регулярных выражений для классификации операций
regular_dict = {
             'комм.пл'    :['(Связь, интернет и ТВ)', '(МОСОБЛГАЗ)','(МОСЭНЕРГОСБЫТ)','(РОСТЕЛЕКОМ)','(Кэшбэк СБП)',
                            '(билайн)', '(WEB_SBERBANK_ONL@IN_PAY)', '(Коммунальные\sплатежи)','(Мосэнергосбыт)','(Ростелеком)',
                            '(MAPP_SBERBANK_ONL@IN_PAY)','(На счёт в другой банк)','(ПАО "МТС-БАНК")','(Исх. перевод по номеру счёта)']
            ,'подписки'   :['(Цифровые товары)',	
                            '(IVI )','(Start.ru)','(START.RU)','(YM\*KASPERSKY\sMOSCOW)','(YANDEX PLUS)','(PODPISKA PREMIUM)']
            ,'Б %'        :[]
            ,'Б усл'      :['(MOBILE\sBANK\:\sKOMISSIYA)','(Regular\sCharge)'] 
            ,'кредит'     :['(Кредиты)']
            ,'супер.м'    :['(Продукты)', '(Алкоголь)',
                            '(PEREKRESTOK)', '(Супермаркеты)', '(VKUSVILL)','(MOSCOW.{0,1}VV)','(TASTY COFFEE)','(MAGNOLIYA)',
                            '(MAGNIT)','(MYASNOJ\sPASSAZH)','(ZHITNAYA\s10)','(EUROSPAR)','(KRASNAYA\sIKRA)',
                            '(Выплата\sкэшбэка)','(PYATEROCHKA)','(IP\sKAZAKOV)','(Yandex Delivery)','(METRO STORE)',
                            '(DOSTAVKA IZ PYATEROCH)','(Выплата за операции покупок)','(TC PAVELECKAYA)','(Табак)']
            ,'ReCa'       :['(Фастфуд)',	
                            '(Рестораны и кафе)','(AVRORA RUSKO)','(M.FOOD\sZALY)','(KAPUCHINOFF)','(M FOOD)', '(Yakitoriya)','(TRUE COST)',
                           '(OSTERIA MARIO)']
            ,'маркет.п'   :['(Маркетплейсы)',	
                            '(OZON)','(WILDBERRIES)']
            ,'Од&О'       :['(STOLICHNYJ GARDEROB)']
            ,'Кр&З'       :['(KRASNOPRESNENSKIE\sBAN)','(GOLD\sAPPLE)','(BARBERSHOP\sBRITVA)','(SALON KRASOTY)','(BRITVA)',
                            '(NUSRATULLO MUKHAMADIE)']
            ,'мед&ап'     :['(Аптеки)','(Медицинские услуги)',
                            '(VASHA\s1)','(APTEKAASTRA)']
            ,'бензин'     :['(АЗС)',
                            '(MOSCOW RNAZK)','(AZS)']
            ,'авто'       :['(AVTODOR MOSCOW)','(AMPP)']
            ,'TAXI'       :['(Такси	)',
                           '(YANDEX GO)','(YandexGo)']
            ,'трансп'     :['(Транспорт)',
                            '(Moskva Metro)']
            ,'Дет&канц'   :['(BELYY\sKROLIK )','(BELYJ\sKROLIK)']
            ,'Дом&рем'    :['(MOSCOW OBI)', '(MOSCOW MAISON DE LA)','(IP ANASHKIN S V)','(IP.ANASHKIN)','(IP Gulbina G G)','(Дом и ремонт)']
            ,'развл&экс'  :['(Культура и искусство)',
                            '(Отдых\sи\sразвлечения)','(DOM MUZYKI)']
            ,'тех&под'    :['(Цветы)','(Техника)',
                           '(MAGAZIN TSVETY)','(ООО "СОЛНЕЧНЫЙ СВЕТ")']
            ,'Спорт.т'    :[]
            ,'Отдых'      :['(Путешествия)',
                           '(ANEX TOUR)']
            ,'а/б ж.д'    :[]
            ,'Ек'         :['(Финансовые операции)','(Пополнения)','(Переводы)',
                            '(Kazansky\shram)', '(KHRAM\sFLORA\sI\sLAVRA)','(АО "ТК "ЦЕНТР")',
                            '(Александр М.)','(Александра О.)',
                            '(Перевод\s\w{2,}\s)(?![В.\s{1,}\sАндрей\s{1,}Валерьевич]|[В.\s{0,}Екатерина\s{1,}Александровна])']
            ,'Ан'         :[]
            ,'Ар'         :['(Арина В.)','(FOXFORD)','(foxford)']
            ,'ZOO'        :['(Животные)',
                            '(IN\sGOOD\sHANDS)','(ZOOGUM)']
            ,'невыясн'     :['(Возврат по операции покупки)','(Пополнения)']
            ,'проч'       :['(Прочие\sрасходы)','(NALOG.RU)']
            ,'перевод'    :['(Перевод\s\w{2,}\sВ.\s{0,}\Андрей\s{1,}Валерьевич)', '(Перевод\s\w{2,}\sВ.\s{0,0}Екатерина\sАлександровна)',
                            '(Перевод\sс\sкарты\sдругого\sбанка)','(Перевод\sс\sкарты)', '(Екатерина Александровна В)','(Андрей В.)',
                            '(Между своими счетами)','(Екатерина В.)']
            ,'аренда'     :[]
            ,'з/п'        :['("НК "Роснефть")']
            ,'бонус'      :[]
            ,'нал'        :['(Выдача наличных)',
                            '(ATM)']
            ,'0'          :['(Увеличение\sкредитного\sлимита)','(ВЛАДИМИРОВА ЕКАТЕРИНА АЛЕКСАНДРОВНАПогашение О...)']
            }

import re
import numpy as np
df['key'] = ''
#организуем цикл по столбцу df "Категория"
for i, payment_name in enumerate(df['Категория']): 
    #print(payment_name)
    payment_name.replace('  ',' ')
    for key in regular_dict: 
        #извлекаем шаблоны из словаря  по ключу key (тип операции)
        pattern_list = regular_dict[key]
        for pattern in pattern_list:
            meta_pattern = re.compile(pattern)
            #print(key, pattern, meta_pattern)
            #проверяем наличие шаблона в строке
            key_pattern = meta_pattern.findall(payment_name)
            #print(key, pattern, meta_pattern, key_pattern)
            #присваиваем аттрибут key (тип операции) в строку df_1, если pattern найден 
            if key_pattern:     
                #print(i, key, pattern, meta_pattern, key_pattern)
                ii = i
                if df.loc[i,'Сумма платежа'] == '' :
                    ii = i - 1
                df.loc[ii,'key'] = key   
                break  # Выход из внутреннего цикла, если найдено совпадение

#организуем цикл по столбцу df "Назначение платежа"
for i, payment_name in enumerate(df['Назначение платежа']): 
    #print(payment_name)
    payment_name.replace('  ',' ')
    for key in regular_dict: 
        #извлекаем шаблоны из словаря  по ключу key (тип операции)
        pattern_list = regular_dict[key]
        for pattern in pattern_list:
            meta_pattern = re.compile(pattern)
            #print(key, pattern, meta_pattern)
            #проверяем наличие шаблона в строке
            key_pattern = meta_pattern.findall(payment_name)
            #print(key, pattern, meta_pattern, key_pattern)
            #присваиваем аттрибут key (тип операции) в строку df_1, если pattern найден 
            if key_pattern:     
                #print(i, key, pattern, meta_pattern, key_pattern)
                ii = i
                if df.loc[i,'Сумма платежа'] == '' :
                    ii = i - 1
                df.loc[ii,'key'] = key   
                break  # Выход из внутреннего цикла, если найдено совпадение
df = df[['Дата','Назначение платежа','Категория','Сумма платежа','key']]
pd.set_option('display.max_rows', 200)  # Установите нужное количество строк
df

,Дата,Назначение платежа,Категория,Сумма платежа,key
0,30.05.2026,MOSCOW/IVI.RU,Прочие расходы,-299.00,проч
1,29.05.2026,Кристина Х.,Финансовые операции,-1850.00,Ек
2,29.05.2026,билайн,"Связь, интернет и ТВ",-1100.00,комм.пл
3,28.05.2026,Сусанна Г.,Финансовые операции,-1400.00,Ек
4,27.05.2026,MOSCOW\IVI RU,Прочие расходы,-449.00,подписки
5,25.05.2026,Арина В.,Финансовые операции,-1000.00,Ар
6,21.05.2026,Андрей В.,Финансовые операции,-60000.00,перевод
7,21.05.2026,Арина В.,Финансовые операции,-16000.00,Ар
8,21.05.2026,"ПАО ""НК ""Роснефть""",Пополнения,100.00,з/п
9,21.05.2026,УФК по Тульской области(МИ ФНС РОССИИ ПО УПРАВ...,Пополнения,7409.00,невыясн


In [16]:
#         Сохранение DataFrame в файл Excel для заполнения пустых ячеек
#!pip install openpyxl                       #уже установлены
#!pip install xlsxwriter                     #уже установлены

#          Путь к существующему файлу Excel
#file_path = directory_out + sheet_name + '.xlsx'
file_path = "https://docs.google.com/spreadsheets/d/1US3sIWQCpLCwEIGJU0GwZ9mkMLdfyDmF/export?format=xlsx"
sheet_name_1 =  sheet_name                   # Имя листа, на который будет записан DataFrame
#          Запись DataFrame в существующий файл Excel на определенный лист
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_name_1, index=False)
print(file_path)
# для просмотра файла excel нужно нажать на появившуюся ссылку и открыть файл из окна загрузок

https://docs.google.com/spreadsheets/d/1US3sIWQCpLCwEIGJU0GwZ9mkMLdfyDmF/export?format=xlsx


C:\Users\avlad\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Data Validation extension is not supported and will be removed
  ws_parser.bind_all()


### исполним файл pdf_sber для преобразования выписки sber_05_26.xlsm в excel файл заданного образца sber.xlsm

In [17]:
directory_path = r""
directory_out = r"C:\Users\avlad\OneDrive\Desktop\Jupyter\OUT\\"
sheet_name = "sber"
file_pdf = sheet_name + '_05_26.pdf'

#pip install PyPDF2
import PyPDF2
import pandas as pd
import numpy as np
import re # импортируем модуль re для работы с регулярными выражениями, в чистом Python их нет
# Запускаем цикл
scanning = False
meta_date = re.compile(r'\d{2}\.\d{2}\.\d{4}')
meta_time_code = re.compile(r'\s(\d{2}:\d{2})\s')
meta_amount =  re.compile(r'(\+{0,1}\d{0,3}\s{0,1}\d{0,3}\s{0,1}\d{0,3}\,\d{2})\s{1,4}\d')
meta_end_amount = re.compile(r'\,\d{2}\s{0,4}(\d{1,3}\s{0,1}\d{0,3}\s{0,1}\d{1,3}\,\d{2})')
title_list = ['Дата','Назначение платежа','amount']
date_list = []
all_data = []
df = pd.DataFrame()
key = 0 #ключ для добавления информации по операции из строки без даты и суммы
# Открываем PDF-файл 
file_pdf = directory_path + file_pdf
with open(file_pdf, 'rb') as file:
    reader = PyPDF2.PdfReader(file)
    num_pages = len(reader.pages)
    print(f'Количество страниц: {num_pages}')
    # Извлекаем текст из каждой страницы
    start = 0
    for page in range(num_pages):
        text = reader.pages[page].extract_text()
        #print(text)
        text_list = text.replace('\xa0','').split('\n')
        #print(text_list)
        for str_i in text_list: 
            if str_i.startswith('Расшифровка операций') :
                start = 1
                #print('Расшифровка операций')
            if start == 0 :      
                continue
            #извлекаем информацию из строки по шаблонам
            date = meta_date.findall(str_i)
            amount = meta_amount.findall(str_i)
            #                         фильтрация по текущему месяцу, если не текущий -> задаем amount = ''
            date_str = ''.join(date)
            month = file_pdf[-9:-7]
            if date_str[3:5] != month :
               amount = []
            time_code = meta_time_code.findall(str_i)
            end_amount = meta_end_amount.findall(str_i)
            #print(str_i)
            #print ('date ',date,' amount', amount, ' time_code', time_code,' end_amount', end_amount)
            #проверяем наличие в строке даты и суммы
            if date and amount:     
                date = ''.join(date)
                amount_str = ''.join(amount)
                if amount_str.startswith('+') :
                    amount_str = amount_str[1:]
                else:
                    amount_str = '-'+ amount_str
                date_str = str_i.replace(date, date +'&')\
                                .replace(''.join(amount),'&'+ amount_str)\
                                .replace(''.join(time_code),'')\
                                .replace(''.join(end_amount),'')
                date_list = date_str.split('&')
                #date_df = pd.DataFrame(date_list)
                #print(date_list)
                key = 1
                scanning = True
            elif key == 1 and date and not amount:
                date_str = str_i.replace(''.join(date),''.join(date)+'&')\
                        +'&'
                date_list = date_str.split('&')
                key = 1
                scanning = True
            elif key == 1 and not date and not amount:
                date_str = '&' + str_i + '&'
                date_list = date_str.split('&')  
                key = 0
                scanning = True
            else:
                scanning = False
                key = 0
            # добавляем строки с информацией по операциям
            if scanning:
                all_data.append(date_list)

if len(all_data) == 0:
    all_data = [[' ', ' ', ' ']]          
df = pd.DataFrame(all_data)
df = df.rename(columns={ 0 : 'Дата',
                         1 : 'Назначение платежа',
                         2 : 'Сумма платежа'})    
#print(df)

regular_dict = {
             'комм.пл'    :['(WEB_SBERBANK_ONL@IN_PAY)','(Коммунальные\sплатежи)','(Мосэнергосбыт)','(Ростелеком)','(MAPP_SBERBANK_ONL@IN_PAY)',
                           '(Филиал Восток_SBP)']
            ,'подписки'   :['(Start.ru)','(START.RU)','(YM\*KASPERSKY\sMOSCOW)','(wgames.pro)']
            ,'Б %'        :[]
            ,'Б усл'      :['(MOBILE\sBANK\:\sKOMISSIYA)','(Regular\sCharge)'] 
            ,'кредит'     :[]
            ,'супер.м'    :['(PEREKRESTOK)', '(Супермаркеты)', '(VKUSVILL)','(MOSCOW VV)','(TASTY COFFEE)','(MAGNOLIYA)',
                            '(MAGNIT)','(MYASNOJ\sPASSAZH)','(ZHITNAYA\s10)','(EUROSPAR)','(KRASNAYA\sIKRA)',
                            '(Выплата\sкэшбэка)','(PYATEROCHKA)','(IP\sKAZAKOV)','(Yandex Delivery)']
                           
            ,'ReCa'       :['(Рестораны и кафе)','(AVRORA RUSKO)','(M.FOOD\sZALY)','(KAPUCHINOFF)','(NOVYJ ATRIUM)']
            ,'маркет.п'   :['(OZON)','(WILDBERRIES)']
            ,'Од&О'       :['(MOSCOW STOLICHNYJ GARDEROB)','(SUNLIGHT)']
            ,'Кр&З'       :['(KRASNOPRESNENSKIE\sBANI)','(GOLD\sAPPLE)','(BARBERSHOP\sBRITVA)','(LAMIONER)','(PODRUZHKA)']
            ,'мед&ап'     :['(VASHA\s1)','(APTEKAASTRA)','(GKB N1 IM. N.I. PIROGOVA)','(APTEKA 55)']
            ,'бензин'     :['(MOSCOW RNAZK)','(AZS)']
            ,'авто'       :['(AVTODOR MOSCOW)','(AMPP MOSCOW)','(cartaxi)','(CARWASH)']
            ,'TAXI'       :['(YANDEX\*{0,1}\d{0,5}\*{0,1}GO)']
            ,'трансп'     :['(Транспорт)', '(Moskva Metro)']
            ,'Дет&канц'   :['(BELYY\sKROLIK )']
            ,'Дом&рем'    :['(MOSCOW OBI)', '(MOSCOW MAISON DE LA)']
            ,'развл&экс'  :['(Отдых\sи\sразвлечения)','(FGBUK_GKD Отмена)']
            ,'тех&под'    :[]
            ,'Спорт.т'    :['(SPORTMARAFON)']
            ,'Отдых'      :[]
            ,'а/б ж.д'    :['(UTAIR)']
            ,'Ек'         :['(Kazansky\shram)', '(KHRAM\sFLORA\sI\sLAVRA)','(Перевод СБП)'
                            ]
            ,'Ан'         :[]
            ,'Ар'         :['(FOXFORD)']
            ,'ZOO'        :['(IN\sGOOD\sHANDS)','(ZOOGUM)']
            ,'невыясн'     :['(Перевод с карты)','(Перевод СБП)','(АЛЬФАСТРАХОВАНИЕ)']
            ,'проч'       :['(NALOG.RU)']
            ,'перевод'    :['(Перевод\s\w{2,}\sВ.\s{0,}\Андрей\s{1,}Валерьевич)',
                            '(Перевод\s\w{2,}\sВ.\s{0,}Екатерина\sАлександровна)',
                            '(Перевод\sс\sкарты\sдругого\sбанка)',
                            '(Альфа-банк. Операция по карте \*\*\*\*6828)',
                            '(АО \"T-Банк\". Операция по карте \*\*\*\*8155)'
                            ]
            ,'аренда'     :['(Перевод для Г. Татьяна Олеговна)']
            ,'з/п'        :['(Пособия по временной нетрудоспособност)']
            ,'бонус'      :[]
            ,'нал'        :['(ATM)']
            ,'0'          :['(Увеличение\sкредитного\sлимита)']
            }

import re
import numpy as np
df['key'] = ''
#организуем цикл по столбцу df "Назначение платежа"
for i, payment_name in enumerate(df['Назначение платежа']): 
    #print(payment_name)
    payment_name.replace('  ',' ')
    for key in regular_dict: 
        #извлекаем шаблоны из словаря  по ключу key (тип операции)
        pattern_list = regular_dict[key]
        for pattern in pattern_list:
            meta_pattern = re.compile(pattern)
            #print(key, pattern, meta_pattern)
            #проверяем наличие шаблона в строке
            key_pattern = meta_pattern.findall(payment_name)
            #print(key, pattern, meta_pattern, key_pattern)
            #присваиваем аттрибут key (тип операции) в строку df_1, если pattern найден 
            if key_pattern:     
                #print(i, key, pattern, meta_pattern, key_pattern)
                ii = i
                if df.loc[i,'Сумма платежа'] == '' :
                    ii = i - 1
                df.loc[ii,'key'] = key   
                break  # Выход из внутреннего цикла, если найдено совпадение
pd.set_option('display.max_rows', 200)  # Установите нужное количество строк
df['Категория'] = ''
df = df[['Дата','Назначение платежа','Категория','Сумма платежа','key']]
df['Категория'] = df.apply(lambda row: row['Назначение платежа'] if row['Сумма платежа'] != '' else '', axis=1)
#         Переносим Категорию из назначения платежа в отдельном поле
for i in range(len(df)-1):
    if df.loc[i,'Назначение платежа'] == df.loc[i,'Категория'] and df.loc[i + 1, 'Сумма платежа'] != ' ' :
        df.loc[i,'Назначение платежа'] = df.loc[i+1,'Назначение платежа']
#           Удаляем лишние строки    
for i in range(len(df)):
    if df.loc[i, 'Сумма платежа'] == '' :
        df.drop(index=i, inplace=True)
df.reset_index(drop=True, inplace=True)

df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce', dayfirst=True).dt.date

df

Количество страниц: 2


,Дата,Назначение платежа,Категория,Сумма платежа,key
0,2026-05-22,193307 Перевод для В. Андрей Валерьевич. Опер...,Перевод с карты,"- 7378,99",перевод
1,2026-05-22,285108 APTEKA 55 MOSCOW RUS. Операция по карт...,Здоровье и красота,"- 792,00",мед&ап
2,2026-05-22,481395 ATM 60029809 MOSKVA RUS. Операция по к...,Выдача наличных,"- 50000,00",нал
3,2026-05-21,191857 Перевод от В. Екатерина Александровна....,Перевод СБП,"60000,00",перевод
4,2026-05-19,341861 ATM 60317872 PAVLOVSKIY PO RUS. Операц...,Выдача наличных,"- 1000,00",нал
5,2026-05-19,107756 Перевод для Л. Олег Анатольевич. Опера...,Перевод с карты,"- 4000,00",невыясн
6,2026-05-19,768177 AZS 50254 BOLSHIE DVORY RUS. Операция ...,Автомобиль,"- 4510,00",бензин
7,2026-05-18,148213 Перевод от В. Екатерина Александровна....,Перевод СБП,"10000,00",перевод
8,2026-05-17,263725 wgames.pro. Операция по карте ****8155,Оплата по QR–коду СБП,"- 1070,00",подписки
9,2026-05-17,640848 wgames.pro. Операция по карте ****8155,Оплата по QR–коду СБП,"- 2568,00",подписки


In [18]:
#         Сохранение DataFrame в файл Excel для заполнения пустых ячеек
#!pip install openpyxl                       #уже установлены
#!pip install xlsxwriter                     #уже установлены

#          Путь к существующему файлу Excel
#file_path = directory_out + sheet_name + '.xlsx'
file_path = "https://docs.google.com/spreadsheets/d/1kmsmMox90NxJzG4w2vOvtRA_zT3GUgbk/export?format=xlsx"
sheet_name_1 =  sheet_name                   # Имя листа, на который будет записан DataFrame
#          Запись DataFrame в существующий файл Excel на определенный лист
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_name_1, index=False)
print(file_path)
# для просмотра файла excel нужно нажать на появившуюся ссылку и открыть файл из окна загрузок

https://docs.google.com/spreadsheets/d/1kmsmMox90NxJzG4w2vOvtRA_zT3GUgbk/export?format=xlsx


C:\Users\avlad\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Data Validation extension is not supported and will be removed
  ws_parser.bind_all()


- ### в случае невыясненных сумм дополнение матрицы регулярных выражений или внесение типа невыясненной операции с помощью выпадающего списка в файле excel на листе "Лист 1"  (вызывается через Разработчик -> Макросы -> Выпадающий список),
- ### формирование итогового листа по группам операций для каждой выписки с использованием макроса excel (нажать на кнопку "Сформировать итоговый файл" на листе "Лист 1"),
- ### создание полного набора excel файлов выписок, 
- ### проверка равенства «0» суммы всех внутренних переводов по всему набору excel файлов выписок (excel файл «переводы»),
- ### формирование сводной таблицы за месяц по всему набору  excel файлов выписок, внесение данных по наличным операциям, кредитам и прочим  долговым обязательствам, внесение или корректировка фактических остатков,
- ### расчет итогового ежемесячного cash flow , PNL  и суммы чистых активов.
### Бюджетирование:
### По сводной таблице за месяц проводится сопоставление фактических и плановых значений по каждому разделу и каждой группе операций.
### Для более глубокого анализа конкретной группы расходов проводится детальный анализ за отчетный и предыдущий месяц с использованием библиотеки Pandas.
### Финансовое планирование:
### Данные сводной таблицы за месяц вносятся в годовой ежемесячный план. Проводятся корректировки и уточнения регулярных и крупных-единовременных операций.


